# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [2]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [1]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [ ]:
# 🤖 AGENT FUNCTION

import json
import re

def agent(query: str):
    """
    Route a query to the appropriate tool and always return:
    {"type": "...", "result": ...}
    """
    if not isinstance(query, str):
        return {
            "type": "error",
            "result": "Query must be a string"
        }

    query_lower = query.lower().strip()

    if not query_lower:
        return {
            "type": "error",
            "result": "Query cannot be empty"
        }

    # 1. Calculation route
    if "calculate" in query_lower:
        expression = query[query_lower.find("calculate") + len("calculate"):].strip()

        if not expression:
            return {
                "type": "error",
                "result": "Missing mathematical expression after 'calculate'"
            }

        if not re.fullmatch(r"[0-9+\-*/%().\s]+", expression):
            return {
                "type": "error",
                "result": "Invalid mathematical expression format"
            }

        result = calculator(expression)

        if result == "Error in calculation":
            return {
                "type": "error",
                "result": result
            }

        return {
            "type": "calculation",
            "result": result
        }

    # 2. Keyword extraction route
    elif "keywords" in query_lower:
        text = query[query_lower.find("keywords") + len("keywords"):].strip()

        if text.lower().startswith("from "):
            text = text[5:].strip()

        if not text:
            return {
                "type": "error",
                "result": "Missing text after 'keywords'"
            }

        return {
            "type": "keywords",
            "result": extract_keywords(text)
        }

    # 3. General fallback route
    else:
        return {
            "type": "general",
            "result": "I can help with calculations or keyword extraction. "
                      "Please use 'calculate' or 'keywords' in your query."
        }


def print_json_response(query, response):
    """Print a response as valid JSON for validation/debugging."""
    print(json.dumps({
        "query": query,
        "response": response
    }, ensure_ascii=False))


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

# Routing and Validation Notes

The agent uses explicit lowercase substring checks in this order:

1. "calculate": extracts the expression and calls calculator.
2. "keywords": passes the text to extract_keywords.
3. Anything else : returns a general fallback.
4. Missing/invalid command content or invalid input → returns error.

Every agent response contains exactly the required type and result keys, and validation output is printed with json.dumps() so it is valid JSON rather than Python dictionary representation.


In [4]:
# 🧪 Automated Validation Checks

validation_queries = [
    ("Calculate 20 + 5", "calculation"),
    ("Calculate (10 + 5) * 2", "calculation"),
    ("Extract keywords from Artificial Intelligence is transforming industries", "keywords"),
    ("keywords machine learning models improve prediction", "keywords"),
    ("What is machine learning?", "general"),
    ("calculate", "error"),
    ("keywords", "error"),
    ("Calculate 10 + abc", "error"),
]

all_passed = True

for query, expected_type in validation_queries:
    response = agent(query)
    passed = (
        isinstance(response, dict)
        and set(response.keys()) == {"type", "result"}
        and response["type"] == expected_type
    )

    all_passed = all_passed and passed

    # Dictionary-valid JSON output for every validation check.
    print(json.dumps({
        "query": query,
        "type": response.get("type"),
        "result": response.get("result"),
        "validation": "PASS" if passed else "FAIL"
    }, ensure_ascii=False))

print(json.dumps({
    "type": "general",
    "result": "All automated validation checks passed" if all_passed
             else "One or more automated validation checks failed"
}, ensure_ascii=False))


{"query": "Calculate 20 + 5", "type": "calculation", "result": "25", "validation": "PASS"}
{"query": "Calculate (10 + 5) * 2", "type": "calculation", "result": "30", "validation": "PASS"}
{"query": "Extract keywords from Artificial Intelligence is transforming industries", "type": "keywords", "result": ["intelligence", "industries", "transforming", "artificial"], "validation": "PASS"}
{"query": "keywords machine learning models improve prediction", "type": "keywords", "result": ["improve", "prediction", "models", "machine", "learning"], "validation": "PASS"}
{"query": "What is machine learning?", "type": "general", "result": "I can help with calculations or keyword extraction. Please use 'calculate' or 'keywords' in your query.", "validation": "PASS"}
{"query": "calculate", "type": "error", "result": "Missing mathematical expression after 'calculate'", "validation": "PASS"}
{"query": "keywords", "type": "error", "result": "Missing text after 'keywords'", "validation": "PASS"}
{"query":

In [ ]:
#Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")

    if user_input.strip().lower() == "exit":
        print(json.dumps({
            "type": "general",
            "result": "Interactive session ended"
        }))
        break

    try:
        response = agent(user_input)
        print(json.dumps(response, ensure_ascii=False))
    except Exception as exc:
        print(json.dumps({
            "type": "error",
            "result": f"Agent error: {str(exc)}"
        }, ensure_ascii=False))


{"type": "error", "result": "Query cannot be empty"}
{"type": "calculation", "result": "25"}
{"type": "calculation", "result": "6.666666666666667"}
{"type": "keywords", "result": ["intern", "celebal", "science"]}
{"type": "keywords", "result": ["intern", "technologies", "celebla", "science"]}
{"type": "error", "result": "Query cannot be empty"}
{"type": "general", "result": "Interactive session ended"}


# Conclusion:

In this task, I implemented a simple task-routing agent using conditional logic on the user query. The agent identifies calculation and keyword-related queries and routes them to the appropriate tools, while other queries are handled by a general response. I also added error handling for invalid or incomplete inputs and verified the agent using multiple test cases and the interactive loop. The final responses follow a consistent type and result structure, making the pipeline easier to test and understand.